# Phase 4 — Modeling

**Inputs:** `data/processed/train.parquet`, `data/processed/test.parquet` (Phase 3).

**Outputs:** `outputs/models/lr_baseline_v1.joblib`, `outputs/models/xgb_v1.joblib`, plus diagnostic figures in `outputs/figures/`.

**Two models, on purpose.**

1. **Logistic regression baseline** — interpretable, fast, gives sign + magnitude per feature. Anything that XGBoost beats this on is a real gain; anything it doesn’t, the simpler model wins by Occam.
2. **XGBoost** — handles missingness natively, captures non-linear interactions, consumes pandas `category` dtype directly via `enable_categorical=True` so we don’t have to one-hot-encode `addr_state` into 50 columns.

**Imbalance handling.** Train default rate is 20.25% — moderate, not severe. SMOTE on 1.1M rows would be expensive and add synthetic noise; we use `class_weight='balanced'` (LR) and `scale_pos_weight=n_neg/n_pos` (XGB), which adjust the loss without resampling.

**Metrics, ordered by importance for credit risk:**

| Metric | What it tells you | Why credit-risk people use it |
|---|---|---|
| **KS** (Kolmogorov–Smirnov) | max gap between score CDFs of defaults vs repays | Standard regulatory metric; threshold-free measure of separation |
| **ROC-AUC** | rank-ordering quality | Easy to compare across models, but flattering on imbalanced data |
| **PR-AUC** | precision at every recall level | Less flattering than ROC-AUC on rare events; better signal for which model actually finds defaulters |
| **Brier score** | mean squared error of probabilities | Calibration matters when downstream pricing/provisioning uses the probability directly, not just the rank |
| **log-loss** | calibration + sharpness | Penalizes confidently wrong predictions |

We report all five on the **time-held-out** test set (issue_year ≥ 2017). The 7-point default-rate jump train→test is real concept drift, so test metrics will look worse than train — that’s the honest number.

## 0. Setup

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_curve, precision_recall_curve, confusion_matrix,
)

PROJECT = Path(r'C:\Users\User\Desktop\Credit Risk Prediction System')
sys.path.insert(0, str(PROJECT / 'src'))

from prepare import load_processed
from models import build_lr, build_xgb, split_xy, evaluate, model_path

FIGS = PROJECT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
train, test = load_processed()
X_train, y_train = split_xy(train)
X_test,  y_test  = split_xy(test)
print(f'train: X={X_train.shape}  default rate {y_train.mean()*100:.2f}%')
print(f'test:  X={X_test.shape}   default rate {y_test.mean()*100:.2f}%')

## 1. Logistic regression baseline

Pipeline: median-impute + standardize numerics, most-frequent-impute + one-hot categoricals, then `LogisticRegression(class_weight='balanced')`. All preprocessing lives inside the pipeline so the imputer’s medians are fit on training data only — no leakage from test.

In [ ]:
t0 = time.time()
lr = build_lr(X_train)
lr.fit(X_train, y_train)
print(f'Fit in {time.time() - t0:.1f}s')

p_lr_train = lr.predict_proba(X_train)[:, 1]
p_lr_test  = lr.predict_proba(X_test)[:, 1]

m_lr_train = evaluate(y_train, p_lr_train)
m_lr_test  = evaluate(y_test,  p_lr_test)
print('TRAIN:', m_lr_train)
print('TEST: ', m_lr_test)

## 2. XGBoost

`enable_categorical=True` lets XGBoost split on pandas `category` dtype directly. We set `scale_pos_weight = n_negative / n_positive` (≈ 3.9) to compensate for imbalance without resampling. Defaults beyond that are reasonable starting points (`max_depth=6`, `learning_rate=0.1`, `n_estimators=400`, `subsample=0.9`); Phase 5 can tune these against a time-aware CV inside train.

In [ ]:
t0 = time.time()
xgb = build_xgb(y_train)
xgb.fit(X_train, y_train)
print(f'Fit in {time.time() - t0:.1f}s')

p_xgb_train = xgb.predict_proba(X_train)[:, 1]
p_xgb_test  = xgb.predict_proba(X_test)[:, 1]

m_xgb_train = evaluate(y_train, p_xgb_train)
m_xgb_test  = evaluate(y_test,  p_xgb_test)
print('TRAIN:', m_xgb_train)
print('TEST: ', m_xgb_test)

## 3. Head-to-head on test

In [ ]:
comparison = pd.DataFrame({
    'lr_baseline_v1': m_lr_test.as_row(),
    'xgb_v1':         m_xgb_test.as_row(),
}).T.round(4)
comparison

## 4. ROC + Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, p in [('LR', p_lr_test), ('XGB', p_xgb_test)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=name)
axes[0].plot([0, 1], [0, 1], '--', color='grey', alpha=0.5, label='chance')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC — test set')
axes[0].legend()

for name, p in [('LR', p_lr_test), ('XGB', p_xgb_test)]:
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].plot(rec, prec, label=name)
axes[1].axhline(y_test.mean(), linestyle='--', color='grey', alpha=0.5,
                label=f'base rate ({y_test.mean()*100:.1f}%)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall — test set')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGS / 'phase4_roc_pr.png', dpi=120)
plt.show()

## 5. Calibration

If we tell the bank a borrower has a 30% chance of default, are 30% of them actually defaulting? A model with great ROC-AUC can still be badly miscalibrated — and downstream pricing uses the probability, not just the rank, so calibration matters.

Note: `class_weight='balanced'` and `scale_pos_weight` both *de-calibrate* the model in exchange for better recall on the minority class. The reliability curves below should lean above the diagonal — predicting higher than reality. Phase 5 will fix this with isotonic / Platt calibration on a held-out slice of train.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name, p in [('LR', p_lr_test), ('XGB', p_xgb_test)]:
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=20, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', label=name)
ax.plot([0, 1], [0, 1], '--', color='grey', alpha=0.5, label='perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction observed default')
ax.set_title('Reliability diagram — test set')
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / 'phase4_calibration.png', dpi=120)
plt.show()

## 6. Score distributions

How separated are the score distributions for defaulters vs repayers? KS is the max gap between these CDFs; this plot is what KS is measuring.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, name, p in [(axes[0], 'LR', p_lr_test), (axes[1], 'XGB', p_xgb_test)]:
    ax.hist(p[y_test == 0], bins=50, alpha=0.5, label='repay', density=True, color='steelblue')
    ax.hist(p[y_test == 1], bins=50, alpha=0.5, label='default', density=True, color='tomato')
    ax.set_xlabel('Predicted P(default)')
    ax.set_title(f'{name} — test score distribution')
    ax.legend()
fig.tight_layout()
fig.savefig(FIGS / 'phase4_score_distributions.png', dpi=120)
plt.show()

## 7. Feature importance

In [ ]:
# XGBoost — gain-based importance, top 20
booster = xgb.get_booster()
imp = pd.Series(booster.get_score(importance_type='gain'))
imp = imp.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(8, 6))
imp.iloc[::-1].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Gain')
ax.set_title('XGBoost — top 20 features by gain')
fig.tight_layout()
fig.savefig(FIGS / 'phase4_xgb_importance.png', dpi=120)
plt.show()

In [ ]:
# LR — top 20 coefficients by absolute value (recall: numerics are standardized)
feat_names = lr.named_steps['pre'].get_feature_names_out()
coefs = pd.Series(lr.named_steps['clf'].coef_[0], index=feat_names)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).head(20).index)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['tomato' if c > 0 else 'steelblue' for c in top.values]
top.iloc[::-1].plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_xlabel('Standardized coefficient (red = increases default risk)')
ax.set_title('LR — top 20 features by |coefficient|')
ax.axvline(0, color='black', linewidth=0.5)
fig.tight_layout()
fig.savefig(FIGS / 'phase4_lr_coefficients.png', dpi=120)
plt.show()

## 8. Confusion matrix at threshold 0.5 — and why 0.5 is the wrong threshold

Default-threshold predictions are mostly here for completeness. At 0.5 with a class-weighted model, the model over-predicts default (because the loss told it to). The right way to pick a threshold is a cost-curve over (false-accept, false-reject) — that’s a Phase 5 task.

In [ ]:
for name, p in [('LR', p_lr_test), ('XGB', p_xgb_test)]:
    yhat = (p >= 0.5).astype(int)
    cm = confusion_matrix(y_test, yhat)
    print(f'{name}  threshold=0.5  predicted default rate={(yhat==1).mean()*100:.1f}%')
    print(pd.DataFrame(cm, index=['true_repay', 'true_default'],
                       columns=['pred_repay', 'pred_default']).to_string())
    print()

## 9. Save model artifacts

In [ ]:
joblib.dump(lr,  model_path('lr_baseline_v1'))
joblib.dump(xgb, model_path('xgb_v1'))
print('Saved:')
print(f'  {model_path("lr_baseline_v1")}  ({model_path("lr_baseline_v1").stat().st_size/1e6:.1f} MB)')
print(f'  {model_path("xgb_v1")}          ({model_path("xgb_v1").stat().st_size/1e6:.1f} MB)')

## 10. Phase 4 findings — what carries into Phase 5

Fill these in *after* running the cells above (numbers in brackets are placeholders that you’ll replace with what the run produced):

1. **Test metrics:** LR `(ROC-AUC=…, PR-AUC=…, KS=…)`. XGB `(ROC-AUC=…, PR-AUC=…, KS=…)`. XGB beats LR by ~`(…)` ROC points / `(…)` KS points — gain is from `(non-linear interactions / categorical splits / handling missingness)`.
2. **Train→test gap:** Both models’ Brier/log-loss are worse on test by `(…)%`. The drop is concept drift, not overfit (LR is far from flexible enough to overfit a 1.1M-row dataset).
3. **Calibration:** Reliability curves lean `(above / below)` the diagonal because `class_weight='balanced'` re-weights the loss. Phase 5: apply isotonic calibration on a held-out slice of train.
4. **Top features (XGB by gain):** `sub_grade`, `int_rate`, `term`, `dti`, `fico_mean`, …  Consistent with LR’s top coefficients — i.e., the models agree on *what matters*, just not on *how to combine it*.
5. **What Phase 5 needs to deliver:** (a) calibrated probabilities, (b) threshold selected against a real cost matrix (false-accept costs more than false-reject in lending), (c) a stability check that the model degrades gracefully across 2017 and 2018 separately.

---

### Check-for-understanding

1. Why is **KS** more useful than accuracy for choosing between LR and XGB here?
2. Why did we use `class_weight='balanced'` instead of SMOTE-resampling the training data?
3. Looking at the reliability diagram, in which probability range is the model most miscalibrated, and what business decisions would that hurt most?
4. Suppose you read a colleague’s notebook that reports `ROC-AUC=0.99` on this dataset. Name two specific bugs you’d check for before believing the number.
5. The 7-point drop in default rate from train (20.25%) to test (27.20%) is concept drift. Which of the two models is more vulnerable to it, and why?